# %% [markdown]
# ============================================================
# STEP 1 — Overview (Markdown Description)
# ============================================================
# This script processes EPC simulation outputs for multiple buildings.
#
# For each building folder:
#   1. It loads "annual_results.csv" from the TOTAL folder.
#   2. For each row type:
#        - energy
#        - carbon
#        - fixed_cost
#        - flex_cost
#   3. It sums all relevant fuel columns for each timestep (_1 to _9).
#
#   SPECIAL RULE:
#   - For the "energy" row ONLY:
#       Columns beginning with "electricity_exp_" are EXCLUDED from the sum.
#
#   4. It writes the summed values into "annual_aggregate.csv":
#        - Column "1" gets sum of *_1 columns
#        - Column "2" gets sum of *_2 columns
#        - ...
#        - Column "9" gets sum of *_9 columns
#
# The script loops through all building folders automatically.
#
# Assumptions:
# - Folder structure is consistent
# - annual_results.csv contains rows indexed by:
#       energy, carbon, fixed_cost, flex_cost
# - annual_aggregate.csv already exists and has correct structure
#
# Output:
# - Updates annual_aggregate.csv in-place for each building
# ============================================================

In [1]:
# %%
# ============================================================
# STEP 2 — Import Libraries & Define Paths
# ============================================================

import os
import pandas as pd

# Base directory containing all building folders
BASE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Rows we want to process
TARGET_ROWS = ["energy", "carbon", "fixed_cost", "flex_cost"]

# Fuel column prefixes (per timestep)
FUEL_COLUMNS = [
    "electricity_exp",
    "electricity_tot",
    "lpg_tot",
    "mineral_wood_tot",
    "mains_gas_tot",
    "oil_tot",
    "wood_logs_tot",
    "wood_pellets_tot",
    "wood_chips_tot",
    "coal_tot"
]

print("Setup complete.")

Setup complete.


In [2]:
# %%
# ============================================================
# STEP 3 — Function to Process One Building (UPDATED LOGIC)
# ============================================================

def process_building(building_path):
    total_path = os.path.join(building_path, "TOTAL")
    
    results_file = os.path.join(total_path, "annual_results.csv")
    aggregate_file = os.path.join(total_path, "annual_aggregate.csv")
    
    # Skip if required files don't exist
    if not os.path.exists(results_file) or not os.path.exists(aggregate_file):
        print(f"Skipping (missing files): {building_path}")
        return
    
    # Load data
    results_df = pd.read_csv(results_file)
    aggregate_df = pd.read_csv(aggregate_file)
    
    # Ensure 'label' column is index
    if "label" in aggregate_df.columns:
        aggregate_df.set_index("label", inplace=True)
    
    results_df.set_index(results_df.columns[0], inplace=True)
    
    # Process each target row
    for row in TARGET_ROWS:
        if row not in results_df.index:
            print(f"Missing row '{row}' in {building_path}")
            continue
        
        for i in range(1, 10):  # for columns _1 to _9
            
            # Build list of columns dynamically
            cols_to_sum = []
            
            for fuel in FUEL_COLUMNS:
                col_name = f"{fuel}_{i}"
                
                if col_name not in results_df.columns:
                    continue
                
                # EXCLUDE electricity_exp ONLY for "energy"
                if row == "energy" and fuel == "electricity_exp":
                    continue
                
                cols_to_sum.append(col_name)
            
            total_value = results_df.loc[row, cols_to_sum].sum()
            
            # Write into aggregate
            if row in aggregate_df.index:
                aggregate_df.loc[row, str(i)] = total_value
    
    # Reset index and save
    aggregate_df.reset_index(inplace=True)
    aggregate_df.to_csv(aggregate_file, index=False)
    
    print(f"Processed: {os.path.basename(building_path)}")

In [3]:
# %%
# ============================================================
# STEP 4 — Loop Through All Buildings
# ============================================================

# Get all building folders
building_folders = [
    os.path.join(BASE_DIR, folder)
    for folder in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, folder))
]

print(f"Found {len(building_folders)} buildings.")

# Process each building
for building in building_folders:
    process_building(building)

print("All buildings processed.")

Found 117 buildings.
Processed: 1001664924
Processed: 1001829085
Processed: 1001063639
Processed: 1001829071
Processed: 1234761002
Processed: 1002539407
Processed: 1001063637
Processed: 1001664941
Processed: 1001991633
Processed: 1235057414
Processed: 1001829079
Processed: 1001664922
Processed: 1234761003
Processed: 1001664925
Processed: 1000238907
Processed: 1234761004
Processed: 1002313096
Processed: 1001870878
Processed: 1001664940
Processed: 1234982990
Processed: 1234806523
Processed: 1001870876
Processed: 1001870882
Processed: 1002143357
Processed: 1001951902
Processed: 1234621926
Processed: 1234647955
Processed: 1001906269
Processed: 1001951903
Processed: 1001627539
Processed: 1002143351
Processed: 1001627541
Processed: 1236594950
Processed: 1001627570
Processed: 1002031280
Processed: 1001627549
Processed: 1001627540
Processed: 1001627547
Processed: 1001870854
Processed: 1001430744
Processed: 1234760995
Processed: 1235812262
Processed: 1000336709
Processed: 1001991628
Processed: 

In [4]:
# %%
# ============================================================
# STEP 5 — (Optional) Basic Validation Check
# ============================================================

import random

sample_building = random.choice(building_folders)
sample_file = os.path.join(sample_building, "TOTAL", "annual_aggregate.csv")

print(f"Checking: {sample_building}")

df_check = pd.read_csv(sample_file)
print(df_check.head())

Checking: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627568
        label             1             2             3             4  \
0      energy  5.077061e+07  1.887658e+07  1.887658e+07  1.887658e+07   
1      carbon  6.533736e+03  2.144589e+03  2.144589e+03  2.144589e+03   
2  fixed_cost  4.223373e+03  2.642967e+03  2.642967e+03  2.642967e+03   
3   flex_cost  3.836039e+03  2.255634e+03  2.255634e+03  2.255634e+03   

              5             6             7             8             9  
0  1.887658e+07  2.645352e+07  2.336800e+07  2.336800e+07  2.336800e+07  
1  2.144589e+03  2.514596e+03  2.054853e+03  2.054853e+03  2.054853e+03  
2  2.642967e+03  4.035795e+03  3.883370e+03  3.883370e+03  3.883370e+03  
3  2.255634e+03  3.349652e+03  3.197227e+03  3.197227e+03  3.197227e+03  
